In [ ]:
from pathlib import Path
import glob
import os
import re

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
import pandas as pd

In [ ]:
# Input data paths
country_thresholds_path = "../data/country_thresholds.csv"
wb_income_classification_path = "../data/wb_income_classification.csv"
results_dir = "/data/tc-grid/open-gira/results"

# Output paths
plot_dir = Path("../plots")

# Mapping from open-gira STORM_SET to display name
atmospheres_to_scenario = {
    "STORM_constant": "2000 STORM",
    "STORM_CNRM-CM6-1-HR": "2050 STORM RCP8.5 CNRM",
    "STORM_CMCC-CM2-VHR4": "2050 STORM RCP8.5 CMCC",
    "STORM_EC-Earth3P-HR": "2050 STORM RCP8.5 EC-Earth",
    "STORM_HadGEM3-GC31-HM": "2050 STORM RCP8.5 HadGEM3",
    "IRIS_PRESENT": "2000 IRIS",
    "IRIS_SSP5-2050": "2050 IRIS SSP5-8.5",
    "CHAZ_SSP-585_GCM-UKESM1-0-LL_epoch-2010": "2010 CHAZ UKESM1",
    "CHAZ_SSP-585_GCM-CESM2_epoch-2010": "2010 CHAZ CESM2",
    "CHAZ_SSP-585_GCM-CNRM-CM6-1_epoch-2010": "2010 CHAZ CNRM",
    "CHAZ_SSP-585_GCM-EC-Earth3_epoch-2010": "2010 CHAZ EC-Earth3",
    "CHAZ_SSP-585_GCM-MIROC6_epoch-2010": "2010 CHAZ MIROC6",
    "CHAZ_SSP-585_GCM-UKESM1-0-LL_epoch-2050": "2050 CHAZ SSP5-8.5 UKESM1",
    "CHAZ_SSP-585_GCM-CESM2_epoch-2050": "2050 CHAZ SSP5-8.5 CESM2",
    "CHAZ_SSP-585_GCM-CNRM-CM6-1_epoch-2050": "2050 CHAZ SSP5-8.5 CNRM",
    "CHAZ_SSP-585_GCM-EC-Earth3_epoch-2050": "2050 CHAZ SSP5-8.5 EC-Earth3",
    "CHAZ_SSP-585_GCM-MIROC6_epoch-2050": "2050 CHAZ SSP5-8.5 MIROC6",
    "emanuel_ssp-585_gcm-cesm2_epoch-2005": "2005 Emanuel CESM2",
    "emanuel_ssp-585_gcm-cnrm6_epoch-2005": "2005 Emanuel CNRM6",
    "emanuel_ssp-585_gcm-ecearth6_epoch-2005": "2005 Emanuel EC-Earth6",
    #"emanuel_ssp-585_gcm-fgoals_epoch-2005": "2005 Emanuel FGOALS",
    #"emanuel_ssp-585_gcm-ipsl6_epoch-2005": "2005 Emanuel IPSL6",
    "emanuel_ssp-585_gcm-miroc6_epoch-2005": "2005 Emanuel MIROC6",
    #"emanuel_ssp-585_gcm-mpi6_epoch-2005": "2005 Emanuel MPI6",
    #"emanuel_ssp-585_gcm-mri6_epoch-2005": "2005 Emanuel MRI6",
    "emanuel_ssp-585_gcm-ukmo6_epoch-2005": "2005 Emanuel UKMO6",
    "emanuel_ssp-585_gcm-cesm2_epoch-2050": "2050 Emanuel CESM2",
    "emanuel_ssp-585_gcm-cnrm6_epoch-2050": "2050 Emanuel CNRM6",
    "emanuel_ssp-585_gcm-ecearth6_epoch-2050": "2050 Emanuel EC-Earth6",
    #"emanuel_ssp-585_gcm-fgoals_epoch-2050": "2050 Emanuel FGOALS",
    #"emanuel_ssp-585_gcm-ipsl6_epoch-2050": "2050 Emanuel IPSL6",
    "emanuel_ssp-585_gcm-miroc6_epoch-2050": "2050 Emanuel MIROC6",
    #"emanuel_ssp-585_gcm-mpi6_epoch-2050": "2050 Emanuel MPI6",
    #"emanuel_ssp-585_gcm-mri6_epoch-2050": "2050 Emanuel MRI6",
    "emanuel_ssp-585_gcm-ukmo6_epoch-2050": "2050 Emanuel UKMO6",
}
# Specify group members (GCM variants) of track sets
gcm_groups = {
    "IRIS baseline": ["2000 IRIS"],
    "IRIS SSP5-8.5 2050": ["2050 IRIS SSP5-8.5"],
    "STORM baseline": ["2000 STORM"],
    "STORM RCP8.5 2050": [
        "2050 STORM RCP8.5 CNRM",
        "2050 STORM RCP8.5 CMCC",
        "2050 STORM RCP8.5 EC-Earth",
        "2050 STORM RCP8.5 HadGEM3"
    ],
    "CHAZ baseline": [
        "2010 CHAZ UKESM1",
        "2010 CHAZ CESM2",
        "2010 CHAZ CNRM",
        "2010 CHAZ EC-Earth3",
        "2010 CHAZ MIROC6",
    ],
    "CHAZ SSP5-8.5 2050": [
        "2050 CHAZ SSP5-8.5 UKESM1",
        "2050 CHAZ SSP5-8.5 CESM2",
        "2050 CHAZ SSP5-8.5 CNRM",
        "2050 CHAZ SSP5-8.5 EC-Earth3",
        "2050 CHAZ SSP5-8.5 MIROC6",
    ],   
    "Emanuel baseline": [
        "2005 Emanuel CESM2",
        "2005 Emanuel CNRM6",
        "2005 Emanuel EC-Earth6",
        #"2005 Emanuel FGOALS",
        #"2005 Emanuel IPSL6",
        "2005 Emanuel MIROC6",
        #"2005 Emanuel MPI6",
        #"2005 Emanuel MRI6",
        "2005 Emanuel UKMO6",
    ],
    "Emanuel SSP5-8.5 2050": [
        "2050 Emanuel CESM2",
        "2050 Emanuel CNRM6",
        "2050 Emanuel EC-Earth6",
        #"2050 Emanuel FGOALS",
        #"2050 Emanuel IPSL6",
        "2050 Emanuel MIROC6",
        #"2050 Emanuel MPI6",
        #"2050 Emanuel MRI6",
        "2050 Emanuel UKMO6",
    ],
}
group_colours = {
    "IRIS": "darkgreen",
    "STORM": "purple",
    "CHAZ": "red",
    "Emanuel": "blue",
}

In [ ]:
# Assemble optimum country to threshold mapping
# Calibrated thresholds, ms-1
per_country_calibrated_thresholds = pd.read_csv(country_thresholds_path).rename(columns={"iso_a3": "ISO_A3"}).set_index("ISO_A3")
# Thresholds based on income, ms-1
income_threshold_map = {"L": "30.0", "LM": "30.0", "UM": "30.0", "H": "32.5"}
thresholds = pd.read_csv(wb_income_classification_path, comment="#").set_index("ISO_A3")
thresholds["threshold_ms-1"] = thresholds.INCOME_LEVEL.map(income_threshold_map)
thresholds.loc[per_country_calibrated_thresholds.index, "threshold_ms-1"] = per_country_calibrated_thresholds["threshold_ms-1"].astype(str)

#rps = [i * base for base in (1, 10, 100) for i in range(1, 10)]
rps = np.logspace(0, 3, 100)
baseline = {}
future = {}
for atmosphere, scenario in atmospheres_to_scenario.items():
    
    print(atmosphere)
    paths = glob.glob(os.path.join(results_dir, f"power/by_country/*/disruption/{atmosphere}/pop_affected_by_event.pq"))
    data = {path.split("/")[-4]: pd.read_parquet(path,) for path in paths}
    calibrated_data = {}
    for iso_a3, df in data.items():
        try:
            threshold = thresholds.loc[iso_a3, "threshold_ms-1"]
            calibrated_data[iso_a3] = df.loc[:, str(threshold)]
        except KeyError:
            pass
            #print(f"{iso_a3} missing...")
            
    concat = pd.concat(calibrated_data.values())
    concat = pd.DataFrame(concat).rename(columns={0: "pop_affected"})
    
    global_sum = concat.groupby(concat.index).sum()
    global_sum = global_sum.reset_index()
    if atmosphere.startswith("STORM") or atmosphere.startswith("IRIS"):
        global_sum["year"] = global_sum.event_id.apply(lambda s: int(s.split("_")[2]))
    elif atmosphere.startswith("CHAZ") or atmosphere.startswith("emanuel"):
        global_sum["year"] = global_sum.event_id.apply(lambda s: int(re.search(r"Y(\d{4})", s).groups()[0]))
    annual_max = global_sum.drop(columns=["event_id"]).groupby("year").max()
    df = annual_max.sort_values("pop_affected")
    df = df.reset_index()
    
    # For any atmospheres with more than 1 millenium, drop extra years
    df = df[(df.year >= 0) & (df.year < 1000)]

    # Label with plotting position
    n_years = df.year.max() - df.year.min() + 1
    year_range = range(n_years - 1, -1, -1)
    df["rank"] = year_range
    df["return_period_years"] = (n_years + 1) / df["rank"]

    # Interpolate to common RP grid
    df = df.drop(columns=["year", "rank"])
    df = df[np.isfinite(df.return_period_years)]
    df["log_rp"] = np.log(df.return_period_years)
    target = pd.DataFrame({"return_period_years": rps, "log_rp": np.log(rps), "pop_affected": np.nan})
    df = pd.concat([df, target]).sort_values("log_rp").set_index("log_rp")
    df = df.interpolate(method="index")
    df = df.loc[np.log(rps)].reset_index(drop=True)

    for group_name, members in gcm_groups.items():
        if scenario in members:
            break

    family, *other = group_name.split()
    if other[0] == "baseline":
        baseline[(family, scenario)] = df
    else:
        future[(family, scenario)] = df

In [ ]:
f, axes = plt.subplots(1, 2, figsize=(8,4))
baseline_ax, future_ax = axes
alpha = 0.15
labelsize = 10
iqr_ls = "-"
iqr_lw = 0.5

for ax, dataset in ((baseline_ax, baseline), (future_ax, future)):
    for (family, scenario), df in dataset.items():    
        ax.plot(
            df["return_period_years"],
            df["pop_affected"],
            label=scenario,
            alpha=alpha,
            color=group_colours[family],
        )

        q25 = pd.concat(dataset).groupby("return_period_years").agg(lambda x: np.quantile(x, 0.25)).reset_index()
        ax.plot(q25["return_period_years"], q25["pop_affected"], color="grey", ls=iqr_ls, lw=iqr_lw)
        
        q50 = pd.concat(dataset).groupby("return_period_years").agg(np.median).reset_index()
        ax.plot(q50["return_period_years"], q50["pop_affected"], color="black", lw=1)

        q75 = pd.concat(dataset).groupby("return_period_years").agg(lambda x: np.quantile(x, 0.75)).reset_index()
        ax.plot(q75["return_period_years"], q75["pop_affected"], color="grey", ls=iqr_ls, lw=iqr_lw)

        ax.fill_between(q25["return_period_years"], q25["pop_affected"], q75["pop_affected"], color="grey", alpha=0.1)

# Invisible handle used purely as a spacer/header placeholder
blank = Line2D([0], [0], color="none")
handles = [
    blank,
    *[Line2D([0], [0], color=c, lw=2, alpha=alpha * 2) for c in group_colours.values()],
    blank,
    Line2D([0], [0], color="black", lw=2, alpha=1),
    mpatches.Patch(color='grey', alpha=1),
    #Line2D([0], [0], color="grey", lw=iqr_lw, alpha=1, ls=iqr_ls),
]
labels = [r"$\bf{Model}$", *group_colours.keys(), "", "Median", "IQR"]
future_ax.legend(
    handles, labels,
    loc="lower right", bbox_to_anchor=(1, 0),
    frameon=False, handlelength=2.0, labelspacing=0.6,
    fontsize=9,
)
for ax in axes:
    ax.set_xlim(2, 500)
    ax.set_ylim(8E6, 1.3E8)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.grid(alpha=0.2, which="both")

future_ax.set_yticklabels([])
for ax, text in ((baseline_ax, "Baseline"), (future_ax, "2050 RCP8.5/SSP5-8.5")):
    ax.text(0.04, 0.95, text, fontsize=labelsize, horizontalalignment='left', verticalalignment='top', transform = ax.transAxes)

f.supxlabel("Return period [years]", fontsize=labelsize, y=0.05)
f.supylabel("Population at risk", fontsize=labelsize, x=0.03)
    
plt.subplots_adjust(left=0.1, right=0.95, bottom=0.19, top=0.95, wspace=0.05)
f.savefig(plot_dir / "global_exceedance_spaghetti.pdf")